# Human Activity Classification from Radar using DCNN

## Import Libraries

In [1]:
import os
import re
import glob
import warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import spectrogram as scipy_spectrogram, get_window
from scipy.ndimage import zoom
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, f1_score)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')


e:\Studies\Masters\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Import data as 80-20% Train-Test split format

1. Collect file paths into `records` — each entry is a tuple of `(filepath, subject_id, activity, repetition)`, not the actual signal data.
2. Split the `records` list (i.e. split the file paths) into `train_records` and `test_records`.
3. Load the actual data from disk later, separately for the train and test sets.

In [2]:
# Folder where the dataset is present 
DATA_ROOT          = 'E:\\Studies\\Masters\\Q4\\Object Classification using Radar\\Project\\Dataset_848'
TEST_SUBJECTS_FRAC = 0.20 # fraction of subjects to reserve for testing (20%)
RANDOM_SEED        = 42 # for reproducibility

ACTIVITY_NAMES = {
    1: 'Walking', 2: 'Sitting', 3: 'Standing',
    4: 'Drink water', 5: 'Pick up', 6: 'Fall'
}

def parse_filename(fname):
    """Extract (subject_id, activity_label, repetition) from filename."""
    base = os.path.splitext(os.path.basename(fname))[0].upper()
    m = re.search(r'P(\d+)A(\d+)R(\d+)', base)
    if m is None:
        return None, None, None
    subject_id    = int(m.group(1))
    activity_code = int(m.group(2))
    repetition    = int(m.group(3))
    if not (1 <= activity_code <= 6):
        return None, None, None
    return subject_id, activity_code, repetition


def load_dat_file(filepath):
    """
    Load a single .dat file and return (radarMatrix, params).

    File layout:
      radarData[0]  = fc  [Hz]
      radarData[1]  = Tsweep [ms]
      radarData[2]  = NTS [samples/chirp]
      radarData[3]  = Bw  [Hz]
      radarData[4:] = complex IQ samples (interleaved I/Q floats)

    Returns matrix of shape (NTS, n_chirps) in column-major order.
    """
    try:
        raw = np.loadtxt(filepath)
    except Exception:
        try:
            with open(filepath, 'r') as f:
                content = f.read().split()
            # Handle MATLAB complex notation e.g. "2087+2011i" or "2087-2011i"
            def parse_token(s):
                s = s.replace('i', 'j').replace('I', 'j')
                return complex(s)
            raw = np.array([parse_token(s) for s in content])
        except Exception as e:
            print(f"  [WARN] Cannot read {filepath}: {e}")
            return None, None

    if len(raw) < 5:
        return None, None

    fc_     = raw[0].real
    tsweep_ = raw[1].real * 1e-3
    nts_    = int(raw[2].real)
    bw_     = raw[3].real
    data    = raw[4:]
    params  = dict(fc=fc_, tsweep=tsweep_, nts=nts_, bw=bw_)

    if len(data) < nts_:
        return None, None

    cdata = data.astype(complex)

    n_chirps = len(cdata) // nts_
    cdata    = cdata[:n_chirps * nts_]
    matrix   = cdata.reshape(nts_, n_chirps, order='F')
    return matrix, params


def discover_files(root):
    """Recursively find all .dat files; return list of (path, sid, act, rep)."""
    records = []
    for fpath in glob.glob(os.path.join(root, '**', '*.dat'), recursive=True):
        sid, act, rep = parse_filename(fpath)
        if sid is None:
            continue
        records.append((fpath, sid, act, rep))
    print(f"Found {len(records)} .dat files under '{root}'")
    return records

def split_train_test(records, test_frac=0.20, random_seed=42):
    """Person-independent split: hold out test_frac of all unique subjects globally."""
    rng = np.random.default_rng(random_seed)

    all_subjects = sorted(set(sid for (_, sid, _, _) in records))
    n_test       = max(1, round(len(all_subjects) * test_frac))
    test_subjects = set(rng.choice(all_subjects, size=n_test, replace=False).tolist())

    train_records = [(fp, sid, act, rep) for (fp, sid, act, rep) in records
                     if sid not in test_subjects]
    test_records  = [(fp, sid, act, rep) for (fp, sid, act, rep) in records
                     if sid in test_subjects]

    print(f"Total subjects: {len(all_subjects)}  |  Test subjects: {n_test}")
    print(f"Train: {len(train_records)} files  |  "
          f"Test: {len(test_records)} files  |  "
          f"Test subjects: {sorted(test_subjects)}")
    return train_records, test_records

def load_all_data(records, desc=''):
    """Load all .dat files into memory. Returns list of dicts with matrix + metadata."""
    loaded = []

    for fpath, sid, act, rep in tqdm(records, desc=desc, unit="file"):
        matrix, params = load_dat_file(fpath)
        if matrix is None:
            continue
        loaded.append({
            'fpath':  fpath,
            'matrix': matrix,
            'params': params,
            'sid':    sid,
            'act':    act,
            'rep':    rep
        })

    print(f"{desc}: loaded {len(loaded)}/{len(records)} files successfully")
    return loaded

In [15]:
records = discover_files(DATA_ROOT)

if not records:
    raise FileNotFoundError(
        f"No .dat files found under '{DATA_ROOT}'. "
        "Set DATA_ROOT to the folder containing your .dat files."
    )

# Get activity distribution (number of files per activity)
act_counts = {}
for _, _, act, _ in records:
    act_counts[act] = act_counts.get(act, 0) + 1

print("Activity distribution (files):")
for act in sorted(act_counts):
    print(f"  {ACTIVITY_NAMES[act]:<15s}: {act_counts[act]}")

print(f"Splitting data (test fraction = {TEST_SUBJECTS_FRAC})...")
train_records, test_records = split_train_test(records, test_frac=TEST_SUBJECTS_FRAC, random_seed=RANDOM_SEED)


Found 1754 .dat files under 'E:\Studies\Masters\Q4\Object Classification using Radar\Project\Dataset_848'
Activity distribution (files):
  Walking        : 312
  Sitting        : 312
  Standing       : 311
  Drink water    : 311
  Pick up        : 311
  Fall           : 197
Splitting data (test fraction = 0.2)...
Total subjects: 72  |  Test subjects: 14
Train: 1394 files  |  Test: 360 files  |  Test subjects: [6, 7, 14, 28, 37, 40, 47, 53, 55, 63, 65, 66, 69, 72]


### Radar config

In [ ]:
# Radar hardware parameters
C        = 3e8        # speed of light [m/s]
FC       = 5.8e9      # carrier frequency [Hz]
TSWEEP_S = 1e-3       # chirp duration [s]
NTS      = 128        # samples per chirp
BW       = 400e6      # bandwidth [Hz]
FS       = NTS / TSWEEP_S   # ADC sampling rate (128000 Hz)
PRF      = 1.0 / TSWEEP_S  # pulse repetition frequency (1000 Hz)

# Range bin selection (matches DataProcessingExample.m)
BIN_LOW  = 9    # 0-indexed  (== bin 10 in 1-indexed MATLAB)
BIN_HIGH = 29   # 0-indexed  (== bin 30 in 1-indexed MATLAB)

# Spectrogram parameters
WIN_SAMPLES     = 200
OVERLAP_FACTOR  = 0.95
OVERLAP_SAMPLES = round(WIN_SAMPLES * OVERLAP_FACTOR)   # 190
PAD_FACTOR      = 4
FFT_POINTS      = PAD_FACTOR * WIN_SAMPLES              # 800

# Tile parameters
OBS_DURATION_S = 2.0
IMG_H = 128
IMG_W = 128

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cuda


### Load data

In [17]:
print("Loading training data...")
train_data = load_all_data(train_records, desc="Train")

print("\nLoading test data...")
test_data = load_all_data(test_records, desc="Test")

Loading training data...


Train: 100%|██████████| 1394/1394 [10:41<00:00,  2.17file/s]


Train: loaded 1394/1394 files successfully

Loading test data...


Test: 100%|██████████| 360/360 [02:08<00:00,  2.80file/s]

Test: loaded 360/360 files successfully


## Preprocessing - IQ Correction and Mean Removal

In [18]:
def iq_correction(i_raw, q_raw):
    """
    IQ imbalance correction via Gram-Schmidt orthogonalisation.

    Steps:
      1. Estimate phase imbalance phi = arcsin(corr(I_norm, Q_norm))
      2. Estimate amplitude imbalance a = std(Q) / std(I)
      3. Correct Q: Q_corr = (Q - I * sin(phi)) / (a * cos(phi))
    """
    i_norm    = i_raw / (np.std(i_raw) + 1e-12)
    q_norm    = q_raw / (np.std(q_raw) + 1e-12)
    cross_corr = np.mean(i_norm * q_norm)
    phi        = np.arcsin(np.clip(cross_corr, -1.0, 1.0))
    a_imb      = np.std(q_raw) / (np.std(i_raw) + 1e-12)
    q_corr     = (q_raw - i_raw * np.sin(phi)) / (a_imb * np.cos(phi) + 1e-12)
    return i_raw + 1j * q_corr


def preprocess_matrix(matrix):
    """Mean removal then IQ correction on a raw Range-Time matrix (NTS, n_chirps)."""
    i_raw = np.real(matrix).ravel(order='F')
    q_raw = np.imag(matrix).ravel(order='F')
    i_raw = i_raw - np.mean(i_raw)
    q_raw = q_raw - np.mean(q_raw)
    corrected = iq_correction(i_raw, q_raw)
    return corrected.reshape(matrix.shape[0], matrix.shape[1], order='F')

def preprocess_all(loaded_data, desc=''):
    """Apply mean removal + IQ correction to all loaded matrices in place."""
    failed = 0
    for entry in tqdm(loaded_data, desc=desc, unit="file"):
        try:
            entry['matrix'] = preprocess_matrix(entry['matrix'])
        except Exception as e:
            print(f"  [WARN] Preprocessing failed for {entry['fpath']}: {e}")
            entry['matrix'] = None
            failed += 1

    success = len(loaded_data) - failed
    print(f"{desc}: preprocessed {success}/{len(loaded_data)} files successfully")

In [19]:
print("Preprocessing training data...")
preprocess_all(train_data, desc="Train")

print("\nPreprocessing test data...")
preprocess_all(test_data, desc="Test")

Preprocessing training data...


Train: 100%|██████████| 1394/1394 [01:40<00:00, 13.93file/s]


Train: preprocessed 1394/1394 files successfully

Preprocessing test data...


Test: 100%|██████████| 360/360 [00:26<00:00, 13.64file/s]

Test: preprocessed 360/360 files successfully


## Feature extraction - Spectrograms

In [20]:
def compute_range_map(matrix, nts):
    """Range profiles via FFT along fast-time; returns positive-range half."""
    win  = np.ones(nts)
    rfft = np.fft.fftshift(np.fft.fft(matrix * win[:, None], axis=0), axes=0)
    return rfft[nts // 2:, :]


def compute_spectrogram(matrix, params):
    """Micro-Doppler spectrogram summed over range bins BIN_LOW to BIN_HIGH."""
    nts_    = params.get('nts',    NTS)
    tsweep_ = params.get('tsweep', TSWEEP_S)
    prf_    = 1.0 / tsweep_
    ham_win = get_window('hamming', WIN_SAMPLES)

    rmap     = compute_range_map(matrix, nts_)
    spec_sum = None
    bin_low  = min(BIN_LOW,  rmap.shape[0] - 1)
    bin_high = min(BIN_HIGH, rmap.shape[0] - 1)

    for rb in range(bin_low, bin_high + 1):
        row = rmap[rb, :]
        f, t, Sxx = scipy_spectrogram(
            row, fs=prf_, window=ham_win,
            nperseg=WIN_SAMPLES, noverlap=OVERLAP_SAMPLES,
            nfft=FFT_POINTS, return_onesided=False, mode='complex'
        )
        Sxx_shifted = np.fft.fftshift(np.abs(Sxx), axes=0)
        spec_sum    = Sxx_shifted if spec_sum is None else spec_sum + Sxx_shifted

    if spec_sum is None:
        raise RuntimeError("No valid range bins found.")

    return np.flipud(spec_sum), t, np.fft.fftshift(f)


def spectrogram_to_tiles(spec, t_axis, obs_duration=OBS_DURATION_S,
                          tile_overlap=0.5, img_h=IMG_H, img_w=IMG_W):
    """Slice spectrogram into fixed-duration tiles, resize to (img_h, img_w)."""
    if len(t_axis) < 2:
        return []

    dt          = t_axis[1] - t_axis[0]
    tile_frames = max(1, int(round(obs_duration / dt)))
    step_frames = max(1, int(round(tile_frames * (1 - tile_overlap))))
    n_frames    = spec.shape[1]

    tiles = []
    start = 0
    while start + tile_frames <= n_frames:
        tile    = spec[:, start:start + tile_frames]
        p1, p99 = np.percentile(tile, [1, 99])
        tile    = np.clip((tile - p1) / (p99 - p1 + 1e-12), 0.0, 1.0)
        tile    = zoom(tile, (img_h / tile.shape[0], img_w / tile.shape[1]), order=1)
        tiles.append(tile.astype(np.float32))
        start  += step_frames

    return tiles


def extract_features(loaded_data, desc=''):
    """
    Feature extraction from preprocessed data.
    Expects loaded_data entries to already have preprocessed matrices.
    Returns X (N, 1, IMG_H, IMG_W) and y (N,).
    """
    all_tiles, all_labels = [], []

    for entry in tqdm(loaded_data, desc=desc, unit="file"):
        if entry['matrix'] is None:
            continue

        try:
            spec, t_axis, _ = compute_spectrogram(entry['matrix'], entry['params'])
        except Exception as e:
            print(f"  [WARN] Spectrogram failed for {entry['fpath']}: {e}")
            continue

        tiles = spectrogram_to_tiles(spec, t_axis)
        if not tiles:
            continue

        all_tiles.extend(tiles)
        all_labels.extend([entry['act'] - 1] * len(tiles))

    if not all_tiles:
        raise RuntimeError(f"No tiles extracted for '{desc}'. Check your data.")

    X = np.stack(all_tiles)[:, np.newaxis, :, :]
    y = np.array(all_labels, dtype=np.int64)
    print(f"{desc}: {X.shape[0]} tiles from {len(loaded_data)} recordings")
    return X, y

In [21]:
print("Extracting training features...")
X_train, y_train = extract_features(train_data, desc="Train")

print("\nExtracting test features...")
X_test, y_test = extract_features(test_data, desc="Test")

print(f"\nX_train : {X_train.shape}  y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}   y_test  : {y_test.shape}")

Extracting training features...


Train: 100%|██████████| 1394/1394 [08:33<00:00,  2.71file/s]


Train: 5507 tiles from 1394 recordings

Extracting test features...


Test: 100%|██████████| 360/360 [02:12<00:00,  2.71file/s]

Test: 1400 tiles from 360 recordings

X_train : (5507, 1, 128, 128)  y_train : (5507,)
X_test  : (1400, 1, 128, 128)   y_test  : (1400,)


### Save extracted features in npz file for later use

In [22]:
NPZ_PATH = 'extracted_features.npz'

# ── Save ──
np.savez_compressed(
    NPZ_PATH,
    X_train=X_train, y_train=y_train,
    X_test=X_test,   y_test=y_test
)
print(f"Saved features to '{NPZ_PATH}'")
print(f"  X_train : {X_train.shape}  y_train : {y_train.shape}")
print(f"  X_test  : {X_test.shape}   y_test  : {y_test.shape}")

Saved features to 'extracted_features.npz'
  X_train : (5507, 1, 128, 128)  y_train : (5507,)
  X_test  : (1400, 1, 128, 128)   y_test  : (1400,)


## Classification

### DCNN Model Definition

In [13]:
class ConvBlock(nn.Module):
    """Conv -> BN -> ReLU -> MaxPool -> Dropout block."""
    def __init__(self, in_ch, out_ch, dropout=0.25):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )

    def forward(self, x):
        return self.block(x)


class DCNN(nn.Module):
    def __init__(self, num_classes=6, dropout=0.5,
                 n_conv_blocks=4, base_filters=32, fc_size=512):
        super().__init__()

        layers   = []
        in_ch    = 1
        out_ch   = base_filters
        img_size = 128

        for _ in range(n_conv_blocks):
            layers.append(ConvBlock(in_ch, out_ch, dropout=0.25))
            in_ch    = out_ch
            out_ch   = min(out_ch * 2, 512)
            img_size = img_size // 2

        self.features   = nn.Sequential(*layers)
        flat_size       = in_ch * img_size * img_size

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, fc_size),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(fc_size, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


In [8]:
# Load features from NPZ
NPZ_PATH = 'extracted_features.npz'

data    = np.load(NPZ_PATH)
X_train = data['X_train']
y_train = data['y_train']
X_test  = data['X_test']
y_test  = data['y_test']

print("Loaded features from npz:")
print(f"  X_train : {X_train.shape}  y_train : {y_train.shape}")
print(f"  X_test  : {X_test.shape}   y_test  : {y_test.shape}")

Loaded features from npz:
  X_train : (5507, 1, 128, 128)  y_train : (5507,)
  X_test  : (1400, 1, 128, 128)   y_test  : (1400,)


### Training

In [14]:
# ── Training config ──
# Training parameters
NUM_CLASSES  = 6
BATCH_SIZE   = 32
LR           = 0.000172
EPOCHS_CV    = 100
EPOCHS_FINAL = 100
N_FOLDS      = 5
DROPOUT      = 0.297
WEIGHT_DECAY = 0.000133
PATIENCE     = 20
N_CONV_BLOCKS = 5
BASE_FILTERS  = 64
FC_SIZE       = 512
STEP_SIZE    = 20
GAMMA        = 0.43

ACTIVITY_NAMES = {
    1: 'Walking', 2: 'Sitting', 3: 'Standing',
    4: 'Drink water', 5: 'Pick up', 6: 'Fall'
}

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss  = 0.0
    y_true, y_pred = [], []
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out  = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(X_batch)
        y_true.extend(y_batch.cpu().tolist())
        y_pred.extend(out.argmax(dim=1).cpu().tolist())
    train_loss = running_loss / len(loader.dataset)
    train_acc  = accuracy_score(y_true, y_pred)
    return train_loss, train_acc


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for X_batch, y_batch in loader:
        preds = model(X_batch.to(device)).argmax(dim=1).cpu()
        y_true.extend(y_batch.tolist())
        y_pred.extend(preds.tolist())
    return np.array(y_true), np.array(y_pred)


def run_cv(X, y, device, n_folds=N_FOLDS):
    """5-fold stratified cross-validation with early stopping."""
    skf  = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_accuracies, fold_f1s = [], []
    oof_true, oof_pred        = [], []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n  Fold {fold}/{n_folds}")

        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        tr_loader  = DataLoader(
            TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)),
            batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(
            TensorDataset(torch.tensor(X_val), torch.tensor(y_val)),
            batch_size=BATCH_SIZE)

        model = DCNN(num_classes=NUM_CLASSES, dropout=DROPOUT,
             n_conv_blocks=N_CONV_BLOCKS,
             base_filters=BASE_FILTERS,
             fc_size=FC_SIZE).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)


        best_acc      = 0.0
        no_improve    = 0
        best_weights  = None

        pbar = tqdm(range(EPOCHS_CV), desc=f"  Fold {fold} training", unit="epoch")
        for epoch in pbar:
            loss, train_acc = train_one_epoch(model, tr_loader, criterion, optimizer, device)
            scheduler.step()

            y_true_val, y_pred_val = evaluate(model, val_loader, device)
            val_acc = accuracy_score(y_true_val, y_pred_val)

            pbar.set_postfix(
                loss      = f"{loss:.4f}",
                train_acc = f"{train_acc:.4f}",
                val_acc   = f"{val_acc:.4f}",
                best      = f"{best_acc:.4f}",
                patience  = f"{no_improve}/{PATIENCE}"
            )

            if val_acc > best_acc:
                best_acc     = val_acc
                no_improve   = 0
                best_weights = model.state_dict().copy()
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    print(f"\n  Early stopping at epoch {epoch + 1} "
                          f"(best val acc: {best_acc:.4f})")
                    break

        model.load_state_dict(best_weights)

        y_true_val, y_pred_val = evaluate(model, val_loader, device)
        acc = accuracy_score(y_true_val, y_pred_val)
        f1  = f1_score(y_true_val, y_pred_val, average='macro', zero_division=0)

        fold_accuracies.append(acc)
        fold_f1s.append(f1)
        oof_true.extend(y_true_val.tolist())
        oof_pred.extend(y_pred_val.tolist())

        print(f"  Fold {fold} -- Acc: {acc:.4f}  Macro-F1: {f1:.4f}")

    oof_true = np.array(oof_true)
    oof_pred = np.array(oof_pred)

    print(f"\n  CV mean Acc : {np.mean(fold_accuracies):.4f} +/- {np.std(fold_accuracies):.4f}")
    print(f"  CV mean F1  : {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}")

    return {
        'fold_accuracies':      fold_accuracies,
        'fold_f1s':             fold_f1s,
        'mean_accuracy':        np.mean(fold_accuracies),
        'std_accuracy':         np.std(fold_accuracies),
        'mean_f1':              np.mean(fold_f1s),
        'std_f1':               np.std(fold_f1s),
        'oof_confusion_matrix': confusion_matrix(oof_true, oof_pred)
    }


def train_final(X, y, device):
    """Train final model on all training data with early stopping via a 10% holdout."""
    n_val      = max(1, int(0.1 * len(X)))
    idx        = np.random.default_rng(42).permutation(len(X))
    val_idx    = idx[:n_val]
    tr_idx     = idx[n_val:]

    tr_loader  = DataLoader(
        TensorDataset(torch.tensor(X[tr_idx]), torch.tensor(y[tr_idx])),
        batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(
        TensorDataset(torch.tensor(X[val_idx]), torch.tensor(y[val_idx])),
        batch_size=BATCH_SIZE)

    model = DCNN(num_classes=NUM_CLASSES, dropout=DROPOUT,
             n_conv_blocks=N_CONV_BLOCKS,
             base_filters=BASE_FILTERS,
             fc_size=FC_SIZE).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)

    best_acc     = 0.0
    no_improve   = 0
    best_weights = None

    pbar = tqdm(range(EPOCHS_FINAL), desc="Final model training", unit="epoch")
    for epoch in pbar:
        loss, train_acc = train_one_epoch(model, tr_loader, criterion, optimizer, device)
        scheduler.step()

        y_true_val, y_pred_val = evaluate(model, val_loader, device)
        val_acc = accuracy_score(y_true_val, y_pred_val)

        pbar.set_postfix(
            loss      = f"{loss:.4f}",
            train_acc = f"{train_acc:.4f}",
            val_acc   = f"{val_acc:.4f}",
            best      = f"{best_acc:.4f}",
            patience  = f"{no_improve}/{PATIENCE}"
        )

        if val_acc > best_acc:
            best_acc     = val_acc
            no_improve   = 0
            best_weights = model.state_dict().copy()
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"\n  Early stopping at epoch {epoch + 1} "
                      f"(best val acc: {best_acc:.4f})")
                break

    model.load_state_dict(best_weights)
    print(f"Final model trained -- best val acc: {best_acc:.4f}")
    return model

### Hyperparameter Tuning

In [6]:
# ── Tuning config ──
N_TRIALS      = 100       # number of hyperparameter combinations to try
EPOCHS_TUNE   = 50       # shorter epochs during tuning for speed
PATIENCE_TUNE = 10        # early stopping patience during tuning
VAL_FRAC      = 0.20     # fraction of training data used as validation during tuning

# ── Fixed validation split for tuning (same split every trial for fair comparison) ──
n_val      = max(1, int(VAL_FRAC * len(X_train)))
idx        = np.random.default_rng(42).permutation(len(X_train))
tune_val   = idx[:n_val]
tune_tr    = idx[n_val:]

X_tune_tr  = X_train[tune_tr];  y_tune_tr  = y_train[tune_tr]
X_tune_val = X_train[tune_val]; y_tune_val = y_train[tune_val]

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.25):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )

    def forward(self, x):
        return self.block(x)


class DCNN(nn.Module):
    def __init__(self, num_classes=6, dropout=0.5,
                 n_conv_blocks=4, base_filters=32, fc_size=512):
        super().__init__()

        # ── Conv blocks: filters double each block ──
        layers   = []
        in_ch    = 1
        out_ch   = base_filters
        img_size = 128

        for _ in range(n_conv_blocks):
            layers.append(ConvBlock(in_ch, out_ch, dropout=0.25))
            in_ch     = out_ch
            out_ch    = min(out_ch * 2, 512)   # cap at 512 to avoid explosion
            img_size  = img_size // 2

        self.features   = nn.Sequential(*layers)
        flat_size       = in_ch * img_size * img_size

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, fc_size),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(fc_size, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))
    

def objective(trial):
    # ── Training hyperparameters ──
    lr           = trial.suggest_float('lr',           1e-4, 1e-2, log=True)
    batch_size   = trial.suggest_categorical('batch_size',   [16, 32, 64])
    dropout      = trial.suggest_float('dropout',      0.2,  0.6)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    step_size    = trial.suggest_categorical('step_size',    [5, 10, 20])
    gamma        = trial.suggest_float('gamma',        0.1,  0.7)

    # ── Architecture hyperparameters ──
    n_conv_blocks = trial.suggest_int('n_conv_blocks', 2, 5)
    base_filters  = trial.suggest_categorical('base_filters', [16, 32, 64])
    fc_size       = trial.suggest_categorical('fc_size',      [256, 512, 1024])

    tr_loader  = DataLoader(
        TensorDataset(torch.tensor(X_tune_tr),  torch.tensor(y_tune_tr)),
        batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(
        TensorDataset(torch.tensor(X_tune_val), torch.tensor(y_tune_val)),
        batch_size=batch_size)

    model     = DCNN(num_classes=NUM_CLASSES, dropout=dropout,
                     n_conv_blocks=n_conv_blocks,
                     base_filters=base_filters,
                     fc_size=fc_size).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    best_acc     = 0.0
    no_improve   = 0
    best_weights = None

    for epoch in range(EPOCHS_TUNE):
        train_one_epoch(model, tr_loader, criterion, optimizer, device=DEVICE)
        scheduler.step()

        y_true_val, y_pred_val = evaluate(model, val_loader, device=DEVICE)
        val_acc = accuracy_score(y_true_val, y_pred_val)

        if val_acc > best_acc:
            best_acc     = val_acc
            no_improve   = 0
            best_weights = model.state_dict().copy()
        else:
            no_improve += 1
            if no_improve >= PATIENCE_TUNE:
                break

        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_acc


# ── Run Optuna ──
sampler = TPESampler(seed=42)
study   = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)

optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress per-trial spam

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

# ── Results ──
best = study.best_params
print(f"\nBest trial -- Val Acc: {study.best_value:.4f}")
print(f"Best hyperparameters:")
for k, v in best.items():
    print(f"  {k:<20s}: {v}")

# ── Update config ──
LR            = best['lr']
BATCH_SIZE    = best['batch_size']
DROPOUT       = best['dropout']
WEIGHT_DECAY  = best['weight_decay']
STEP_SIZE     = best['step_size']
GAMMA         = best['gamma']
N_CONV_BLOCKS = best['n_conv_blocks']
BASE_FILTERS  = best['base_filters']
FC_SIZE       = best['fc_size']

print(f"\nTraining config updated with best hyperparameters.")
print(f"  LR           : {LR}")
print(f"  BATCH_SIZE   : {BATCH_SIZE}")
print(f"  DROPOUT      : {DROPOUT}")
print(f"  WEIGHT_DECAY : {WEIGHT_DECAY}")
print(f"  STEP_SIZE    : {STEP_SIZE}")
print(f"  GAMMA        : {GAMMA}")
print(f"  N_CONV_BLOCKS: {N_CONV_BLOCKS}")
print(f"  BASE_FILTERS : {BASE_FILTERS}")
print(f"  FC_SIZE      : {FC_SIZE}")

[I 2026-05-31 15:53:18,201] A new study created in memory with name: no-name-919ba12d-b325-4971-8f11-dbd410177b5a
Best trial: 62. Best value: 0.878292: 100%|██████████| 100/100 [4:32:00<00:00, 163.20s/it] 


Best trial -- Val Acc: 0.8783
Best hyperparameters:
  lr                  : 0.00017222869662454051
  batch_size          : 32
  dropout             : 0.29696959596962613
  weight_decay        : 0.0001326288461650197
  step_size           : 20
  gamma               : 0.43154348252215174
  n_conv_blocks       : 5
  base_filters        : 64
  fc_size             : 512

Training config updated with best hyperparameters.
  LR           : 0.00017222869662454051
  BATCH_SIZE   : 32
  DROPOUT      : 0.29696959596962613
  WEIGHT_DECAY : 0.0001326288461650197
  STEP_SIZE    : 20
  GAMMA        : 0.43154348252215174
  N_CONV_BLOCKS: 5
  BASE_FILTERS : 64
  FC_SIZE      : 512


### 5-fold Cross Validation

In [15]:
# ── Run ──
print("Running 5-fold cross-validation...")
cv_results = run_cv(X_train, y_train, DEVICE)

print("\nTraining final model on all training data...")
final_model = train_final(X_train, y_train, DEVICE)

Running 5-fold cross-validation...

  Fold 1/5


  Fold 1 training:  53%|█████▎    | 53/100 [06:32<05:47,  7.40s/epoch, best=0.8920, loss=0.2047, patience=19/20, train_acc=0.9240, val_acc=0.8820]



  Early stopping at epoch 54 (best val acc: 0.8920)
  Fold 1 -- Acc: 0.8820  Macro-F1: 0.8583

  Fold 2/5


  Fold 2 training:  64%|██████▍   | 64/100 [08:10<04:36,  7.67s/epoch, best=0.8848, loss=0.1659, patience=19/20, train_acc=0.9398, val_acc=0.8820]



  Early stopping at epoch 65 (best val acc: 0.8848)
  Fold 2 -- Acc: 0.8820  Macro-F1: 0.8574

  Fold 3/5


...

  Fold 4 -- Acc: 0.8865  Macro-F1: 0.8625

  Fold 5/5


  Fold 5 training:  95%|█████████▌| 95/100 [11:38<00:36,  7.35s/epoch, best=0.8837, loss=0.1240, patience=19/20, train_acc=0.9580, val_acc=0.8792]



  Early stopping at epoch 96 (best val acc: 0.8837)
  Fold 5 -- Acc: 0.8792  Macro-F1: 0.8537

  CV mean Acc : 0.8789 +/- 0.0075
  CV mean F1  : 0.8543 +/- 0.0078

Training final model on all training data...


Final model training:  94%|█████████▍| 94/100 [12:25<00:47,  7.93s/epoch, best=0.9000, loss=0.1229, patience=19/20, train_acc=0.9552, val_acc=0.8964]


  Early stopping at epoch 95 (best val acc: 0.9000)
Final model trained -- best val acc: 0.9000


## Evaluation

In [16]:
class_names = [ACTIVITY_NAMES[i + 1] for i in range(NUM_CLASSES)]


def evaluate_test(model, X, y, device):
    """Run final model on test set and return metrics."""
    loader = DataLoader(
        TensorDataset(torch.tensor(X), torch.tensor(y)),
        batch_size=BATCH_SIZE)

    y_true, y_pred = evaluate(model, loader, device)

    return {
        'accuracy':       accuracy_score(y_true, y_pred),
        'macro_f1':       f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_per_class':   f1_score(y_true, y_pred, average=None,    zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred),
        'y_true': y_true,
        'y_pred': y_pred
    }


def print_results(results, class_names):
    print(f"\n{'='*60}")
    print(f"  TEST SET RESULTS")
    print(f"{'='*60}")
    print(f"  Overall Accuracy : {results['accuracy']:.4f}  ({results['accuracy']*100:.2f}%)")
    print(f"  Macro F1-Score   : {results['macro_f1']:.4f}")
    print(f"\n  Per-class F1 scores:")
    for i, name in enumerate(class_names):
        print(f"    {name:<15s}: {results['f1_per_class'][i]:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(results['y_true'], results['y_pred'],
                                target_names=class_names, zero_division=0))


def plot_confusion_matrix(cm, title, class_names, filepath=None):
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-12)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(class_names, rotation=30, ha='right', fontsize=9)
    ax.set_yticklabels(class_names, fontsize=9)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            color = 'white' if cm_norm[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{cm_norm[i, j]:.2f}\n({cm[i, j]})',
                    ha='center', va='center', color=color, fontsize=8)
    ax.set_ylabel('True label')
    ax.set_xlabel('Predicted label')
    ax.set_title(title)
    plt.tight_layout()
    if filepath:
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        plt.savefig(filepath, dpi=150, bbox_inches='tight')
        print(f"Saved: {filepath}")
    plt.show()


def plot_cv_accuracy(cv_results, filepath=None):
    accs = cv_results['fold_accuracies']
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(range(1, N_FOLDS + 1), accs, color='steelblue', alpha=0.8)
    ax.axhline(cv_results['mean_accuracy'], color='red', linestyle='--',
               label=f"Mean = {cv_results['mean_accuracy']:.3f}")
    ax.set_xlabel('Fold')
    ax.set_ylabel('Validation Accuracy')
    ax.set_title('5-Fold Cross-Validation Accuracy')
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    if filepath:
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        plt.savefig(filepath, dpi=150, bbox_inches='tight')
        print(f"Saved: {filepath}")
    plt.show()

In [17]:
# ── Run ──
print("Evaluating on test set...")
test_results = evaluate_test(final_model, X_test, y_test, DEVICE)

print_results(test_results, class_names)

plot_cv_accuracy(cv_results,         filepath='results/cv_fold_accuracy.png')
plot_confusion_matrix(cv_results['oof_confusion_matrix'],
                      'CV Out-of-Fold Confusion Matrix',
                      class_names, filepath='results/cv_oof_confusion_matrix.png')
plot_confusion_matrix(test_results['confusion_matrix'],
                      'Test Set Confusion Matrix',
                      class_names, filepath='results/test_confusion_matrix.png')

Evaluating on test set...

  TEST SET RESULTS
  Overall Accuracy : 0.8764  (87.64%)
  Macro F1-Score   : 0.8512

  Per-class F1 scores:
    Walking        : 0.9802
    Sitting        : 0.8661
    Standing       : 0.8672
    Drink water    : 0.7480
    Pick up        : 0.7172
    Fall           : 0.9283

  Classification Report:
              precision    recall  f1-score   support

     Walking       0.99      0.97      0.98       512
     Sitting       0.86      0.87      0.87       189
    Standing       0.89      0.85      0.87       189
 Drink water       0.74      0.76      0.75       186
     Pick up       0.69      0.75      0.72       189
        Fall       0.95      0.91      0.93       135

    accuracy                           0.88      1400
   macro avg       0.85      0.85      0.85      1400
weighted avg       0.88      0.88      0.88      1400

Saved: results/cv_fold_accuracy.png
Saved: results/cv_oof_confusion_matrix.png
Saved: results/test_confusion_matrix.png


In [21]:
os.makedirs('results', exist_ok=True)
torch.save(final_model.state_dict(), 'results/dcnn_final_model.pth')
print("Model saved to results/dcnn_final_model.pth")

print(f"\n{'='*60}")
print(f"  SUMMARY")
print(f"{'='*60}")
print(f"  CV  mean accuracy : {cv_results['mean_accuracy']:.4f} +/- {cv_results['std_accuracy']:.4f}")
print(f"  CV  mean macro-F1 : {cv_results['mean_f1']:.4f} +/- {cv_results['std_f1']:.4f}")
print(f"  Test accuracy     : {test_results['accuracy']:.4f}")
print(f"  Test macro-F1     : {test_results['macro_f1']:.4f}")
print(f"{'='*60}")

Model saved to results/dcnn_final_model.pth

  SUMMARY
  CV  mean accuracy : 0.8466 +/- 0.0136
  CV  mean macro-F1 : 0.8178 +/- 0.0153
  Test accuracy     : 0.8529
  Test macro-F1     : 0.8284
